# MIDI gesture study — Kaggle training60 photos · 120 hand annotations · 3 classes · exactly 40 per class.Four groups of 15 photos at 10/10/10: `fold0`, `fold1`, `fold2`, `test`.A capture place never spans two groups, so no background appears in both train and validation.**Two arms**| arm | geometry | models | runs ||---|---|---|---|| YOLO26 | OBB and axis-aligned | n s m l x | 30 || DEIMv2 | axis-aligned | atto femto pico n s m l x | 24 |**Before running:** Accelerator = **GPU T4 x2** (not P100), Internet **on**,Persistence = **Variables and Files** so a restart resumes instead of starting over.The test set is evaluated **once**, at the end, on the single configuration cross-validationselects. With 54 models and 30 test annotations, choosing on the test set would guarantee aninflated number.

## 1 · Environment checkThirty seconds here saves a wasted session.

In [ ]:
import torch, os, sys, jsonfrom pathlib import Pathprint("torch      ", torch.__version__)print("cuda avail ", torch.cuda.is_available())assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4 x2"name = torch.cuda.get_device_name(0)print("gpu        ", name)assert "P100" not in name, (    "P100 selected. Kaggle's PyTorch targets sm_70+; the P100 is sm_60, so the CUDA kernels "    "are missing and nothing will train. Switch to GPU T4 x2.")# prove kernels actually run, rather than trusting the device nameassert float((torch.zeros(8, device="cuda") + 1).sum()) == 8print("cuda kernels OK")import torchvision; print("torchvision", torchvision.__version__)# Kaggle mounts inputs differently depending on how they were attached: classic datasets# land at /kaggle/input/<slug>/, Data Hub at /kaggle/input/datasets/<user>/<slug>/<slug>/.# Search for the marker files rather than assuming either.ROOT = Path("/kaggle/input")DATA = next((d for d in ROOT.rglob("folds.json")), None)CKPT = next((d for d in ROOT.rglob("deimv2_hgnetv2_n_coco.pth")), None)assert DATA is not None, f"folds.json not found under {ROOT}. Attach the midi-gesture-v2 dataset."assert CKPT is not None, f"checkpoints not found under {ROOT}. Attach deimv2-checkpoints."DATA, CKPT = DATA.parent, CKPT.parentprint("DATA", DATA)print("CKPT", CKPT)print()!ls {DATA}# these two paths feed every later cell%store DATA%store CKPT

## 2 · DependenciesKaggle's image does not ship ultralytics. Installed without touching torch: ultralytics needs`torch>=1.8` and the image has 2.10+cu128, so pip leaves it alone. **If pip reports that it isinstalling or upgrading torch, stop** -- the preinstalled build is matched to the driver andreplacing it breaks CUDA.

In [ ]:
!pip install -q ultralyticsimport ultralytics, torchprint("ultralytics", ultralytics.__version__)print("torch      ", torch.__version__, "| cuda", torch.cuda.is_available())assert torch.cuda.is_available(), "torch lost CUDA - the pip install replaced it"

## 3 · Write the driver scripts

In [ ]:
%%writefile /kaggle/working/train_yolo.py#!/usr/bin/env python3"""YOLO26 training driver for Kaggle. 30 cells: 5 sizes x 2 geometries x 3 folds.RESUME is the point of this script. A Kaggle session can die at 12 hours, or be interrupted,and 30 runs will not always fit in one. Every finished cell appends to results.jsonl in theworking directory; on restart, cells already recorded there are skipped. Put results.jsonl ina Kaggle Dataset (or rely on /kaggle/working persisting between edit sessions) and ahalf-finished sweep continues rather than restarting.PATHS are rebuilt here, never read from the bundle. The split manifests on the source machinehold absolute paths from that machine; a stale manifest makes ultralytics report "0 images found" andtrain on nothing without raising. Regenerating from folds.json means the paths are always theones that exist on this machine.Usage inside a notebook cell:    !python train_yolo.py --data /kaggle/input/midi-gesture-v2 --out /kaggle/working    !python train_yolo.py ... --sizes n s --geoms hbb        # a subset    !python train_yolo.py ... --imgsz-sweep                  # add 320/416 for size n"""from __future__ import annotationsimport argparseimport jsonimport timefrom pathlib import PathCLASS_NAMES = {0: "thumbout", 1: "openhand", 2: "closedhand"}CV_FOLDS = ["fold0", "fold1", "fold2"]def resolve_data(root: Path) -> Path:    """Find the real data root under a Kaggle mount, whatever layout Kaggle used.    Classic datasets mount at /kaggle/input/<slug>/. Data Hub mounts at    /kaggle/input/datasets/<user>/<slug>/<slug>/, and `--dir-mode zip` adds one more level by    preserving the uploaded folder. Rather than encode any of that, search for the marker    files. Getting this wrong yields "0 images found" and a model trained on nothing, so it is    worth being thorough.    """    if not root.exists():        raise SystemExit(            f"{root} does not exist.\n"            f"  Kaggle mounts vary: /kaggle/input/<slug>/ for classic datasets,\n"            f"  /kaggle/input/datasets/<user>/<slug>/<slug>/ for Data Hub.\n"            f"  Find it with:  !find /kaggle/input -name folds.json")    def ok(p: Path) -> bool:        return (p / "folds.json").exists() and (p / "manifest.csv").exists()    if ok(root):        return root    for depth in range(1, 6):        for cand in sorted(root.glob("/".join(["*"] * depth))):            if cand.is_dir() and ok(cand):                print(f"  data root: {cand}")                return cand    raise SystemExit(f"no folds.json + manifest.csv anywhere under {root}"                     f"   try:  !find {root} -name folds.json")def build_splits(data: Path, work: Path, geom: str):    """Write YAMLs and path manifests using paths that exist on THIS machine."""    folds = json.loads((data / "folds.json").read_text())    group_of = folds["photo_group"]    manifest = {}    import csv    with (data / "manifest.csv").open() as fh:        for r in csv.DictReader(fh):            manifest[r["original"].replace(".jpg", "")] = r["id"]    images = data / geom / "images"    members = {g: [] for g in CV_FOLDS + ["test"]}    for stem, grp in group_of.items():        pid = manifest.get(stem)        if pid is None:            raise SystemExit(f"{stem} missing from manifest.csv")        members[grp].append(str(images / f"{pid}.jpg"))    for g in members:        members[g].sort()    sd = work / "splits" / geom    sd.mkdir(parents=True, exist_ok=True)    (sd / "test.txt").write_text("\n".join(members["test"]) + "\n")    yamls = {}    for k, grp in enumerate(CV_FOLDS):        fd = sd / f"fold{k}"        fd.mkdir(exist_ok=True)        (fd / "val.txt").write_text("\n".join(members[grp]) + "\n")        (fd / "train.txt").write_text(            "\n".join(p for o in CV_FOLDS if o != grp for p in members[o]) + "\n")        y = sd / f"fold{k}.yaml"        y.write_text(            f"path: {data / geom}\n"            f"train: {fd / 'train.txt'}\n"            f"val: {fd / 'val.txt'}\n"            f"test: {sd / 'test.txt'}\n"            "names:\n" + "\n".join(f"  {i}: {n}" for i, n in CLASS_NAMES.items()) + "\n")        yamls[grp] = y    return yamlsdef env_fingerprint():    """Record the machine and stack per run.    If any part of a sweep ends up on different hardware -- Kaggle T4 today, a rented 4090    tomorrow -- this is what lets you check whether a difference is the model or the machine.    Training hardware does not change a model to within fold-to-fold noise, but that is a    claim the data should support rather than one the methods section merely asserts.    """    import torch, platform    return {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",            "torch": torch.__version__, "python": platform.python_version(),            "host": platform.node()}def done_cells(path: Path):    if not path.exists():        return set()    out = set()    for line in path.read_text().splitlines():        if not line.strip():            continue        try:            r = json.loads(line)            out.add((r["geom"], r["size"], r["fold"], r["imgsz"]))        except Exception:            pass    return outdef main():    ap = argparse.ArgumentParser()    ap.add_argument("--data", type=Path, required=True)    ap.add_argument("--out", type=Path, default=Path("/kaggle/working"))    ap.add_argument("--sizes", nargs="+", default=["n", "s", "m", "l", "x"])    ap.add_argument("--geoms", nargs="+", default=["hbb", "obb"])    ap.add_argument("--folds", nargs="+", type=int, default=[0, 1, 2])    ap.add_argument("--epochs", type=int, default=100)    ap.add_argument("--imgsz", type=int, default=640)    ap.add_argument("--batch", type=int, default=8)    ap.add_argument("--imgsz-sweep", action="store_true",                    help="also run size n at 320 and 416 (accuracy for the latency arm)")    ap.add_argument("--device", default="0",                    help="cuda index on Kaggle; use mps or cpu to smoke-test locally")    ap.add_argument("--dry-run", action="store_true")    args = ap.parse_args()    args.data = resolve_data(args.data)    from ultralytics import YOLO    import torch    if args.device not in ("cpu", "mps") and torch.cuda.is_available():        name = torch.cuda.get_device_name(0)        print(f"GPU: {name}")        if "P100" in name:            raise SystemExit(                "P100 selected. Kaggle's PyTorch build targets sm_70+ and the P100 is sm_60, "                "so CUDA kernels are missing and nothing will train.\n"                "Switch the accelerator to 'GPU T4 x2'.")        a = torch.zeros(8, device="cuda") + 1        # fail fast rather than 40 runs in        assert float(a.sum()) == 8        print("CUDA kernels OK")    elif args.device in ("cpu", "mps"):        print(f"device={args.device} (local smoke test)")    else:        print("WARNING: no GPU visible")    results = args.out / "results.jsonl"    already = done_cells(results)    print(f"{len(already)} cell(s) already done\n")    cells = []    for geom in args.geoms:        for size in args.sizes:            for k in args.folds:                cells.append((geom, size, k, args.imgsz))    if args.imgsz_sweep:        # sweep whatever --sizes asks for, not a hardcoded "n"        for geom in args.geoms:            for size in args.sizes:                for k in args.folds:                    for sz in (320, 416):                        cells.append((geom, size, k, sz))    todo = [c for c in cells if (c[0], c[1], CV_FOLDS[c[2]], c[3]) not in already]    print(f"{len(cells)} cells total, {len(todo)} to run")    if args.dry_run:        for c in todo:            print("   ", c)        return    yamls = {g: build_splits(args.data, args.out, g) for g in args.geoms}    for i, (geom, size, k, imgsz) in enumerate(todo, 1):        grp = CV_FOLDS[k]        tag = f"{geom}-{size}-fold{k}-{imgsz}"        weights = f"yolo26{size}-obb.pt" if geom == "obb" else f"yolo26{size}.pt"        print(f"\n[{i}/{len(todo)}] {tag}   weights={weights}")        t0 = time.time()        try:            model = YOLO(weights)            model.train(                data=str(yamls[geom][grp]), epochs=args.epochs, imgsz=imgsz,                batch=args.batch, device=args.device, seed=42, deterministic=True,                project=str(args.out / "runs"), name=tag, exist_ok=True,                patience=args.epochs,          # early stopping OFF: with 3 folds it fires at                                               # random and measures luck, not architecture                val=True, plots=False, verbose=False,                # The dataset lives on a read-only mount, so ultralytics cannot save its label                # cache and re-decodes 2238x1492 JPEGs every epoch -- ~20 s/epoch, which makes                # 30 runs exceed a 12 h session. 60 images at 640 px fit in RAM easily.                cache="ram", workers=2,            )            m = model.val(data=str(yamls[geom][grp]), split="val", imgsz=imgsz,                          device=args.device, plots=False, verbose=False)            rec = {                "geom": geom, "size": size, "fold": grp, "imgsz": imgsz,                "epochs": args.epochs, "batch": args.batch,                "mAP50-95": round(float(m.box.map), 6), "mAP50": round(float(m.box.map50), 6),                "precision": round(float(m.box.mp), 6), "recall": round(float(m.box.mr), 6),                "per_class": {CLASS_NAMES[int(c)]: {"ap50": round(float(m.box.ap50[j]), 6),                                                    "ap50_95": round(float(m.box.ap[j]), 6)}                              for j, c in enumerate(m.box.ap_class_index)},                "minutes": round((time.time() - t0) / 60, 2),                "env": env_fingerprint(),                "weights": str(args.out / "runs" / tag / "weights" / "best.pt"),            }        except Exception as e:            rec = {"geom": geom, "size": size, "fold": grp, "imgsz": imgsz,                   "error": f"{type(e).__name__}: {e}",                   "minutes": round((time.time() - t0) / 60, 2)}            print(f"   FAILED: {rec['error']}")        with results.open("a") as fh:            fh.write(json.dumps(rec) + "\n")        if "error" not in rec:            print(f"   mAP50-95 {rec['mAP50-95']:.4f}   mAP50 {rec['mAP50']:.4f}   "                  f"{rec['minutes']:.1f} min")    print(f"\nresults -> {results}")if __name__ == "__main__":    main()

In [ ]:
%%writefile /kaggle/working/setup_deimv2.py#!/usr/bin/env python3"""Prepare DEIMv2 on Kaggle: clone, patch, wire up the data, write configs.DEIMv2's defaults are written for 118k COCO images. On 30 images per fold several of themare not merely suboptimal, they stop training happening at all. This script applies theminimum set of changes and prints what it did, so the paper can state it.  total_batch_size 32 -> 4, and drop_last -> False      floor(30/32) = 0 batches. The dataloader is empty and training silently does nothing.      Open, unanswered DEIM issue #82.  warmup_iter 2000 -> 200      At ~8 iterations per epoch that is 250 epochs of pure warm-up, so the learning rate      never reaches its useful range. The DEIM maintainer diagnosed exactly this in issue #5.  every epoch-indexed schedule rescaled      flat_epoch, no_aug_epoch, policy.epoch, stop_epoch, mixup/copyblend epochs and      matcher_change_epoch are all written against a 148-epoch COCO run.  num_classes 4, not 3      Categories are 1-indexed (1..3), so the head needs 4 slots. DEIMKit documents      classes + 1, and DEIM issue #12 reports num_classes=1 failing where 2 worked.  requirements.txt is NOT installed      It pins torch==2.5.1 / torchvision==0.20.1, which would replace Kaggle's CUDA-matched      build and break every GPU op. Only the genuinely missing packages are installed.Usage:    !python setup_deimv2.py --data /kaggle/input/midi-gesture-v2 \                            --ckpt /kaggle/input/deimv2-checkpoints --work /kaggle/working"""from __future__ import annotationsimport argparseimport jsonimport reimport shutilimport subprocessimport sysfrom pathlib import PathREPO_URL = "https://github.com/Intellindust-AI-Lab/DEIMv2.git"VARIANTS = {    "atto": "deimv2_hgnetv2_atto_coco.pth", "femto": "deimv2_hgnetv2_femto_coco.pth",    "pico": "deimv2_hgnetv2_pico_coco.pth", "n": "deimv2_hgnetv2_n_coco.pth",    "s": "deimv2_dinov3_s_coco.pth", "m": "deimv2_dinov3_m_coco.pth",    "l": "deimv2_dinov3_l_coco.pth", "x": "deimv2_dinov3_x_coco.pth",}GROUPS = ["fold0", "fold1", "fold2", "test"]def resolve_data(root: Path) -> Path:    """Find the real data root under a Kaggle mount, whatever layout Kaggle used.    Classic datasets mount at /kaggle/input/<slug>/. Data Hub mounts at    /kaggle/input/datasets/<user>/<slug>/<slug>/, and `--dir-mode zip` adds one more level by    preserving the uploaded folder. Rather than encode any of that, search for the marker    files. Getting this wrong yields "0 images found" and a model trained on nothing, so it is    worth being thorough.    """    if not root.exists():        raise SystemExit(            f"{root} does not exist.\n"            f"  Kaggle mounts vary: /kaggle/input/<slug>/ for classic datasets,\n"            f"  /kaggle/input/datasets/<user>/<slug>/<slug>/ for Data Hub.\n"            f"  Find it with:  !find /kaggle/input -name folds.json")    def ok(p: Path) -> bool:        return (p / "folds.json").exists() and (p / "manifest.csv").exists()    if ok(root):        return root    for depth in range(1, 6):        for cand in sorted(root.glob("/".join(["*"] * depth))):            if cand.is_dir() and ok(cand):                print(f"  data root: {cand}")                return cand    raise SystemExit(f"no folds.json + manifest.csv anywhere under {root}"                     f"   try:  !find {root} -name folds.json")def run(cmd, **kw):    print("  $", " ".join(str(c) for c in cmd))    return subprocess.run(cmd, check=False, **kw)def stage_data(data: Path, work: Path):    """COCO tree DEIMv2 expects: images/<group>/ + annotations/instances_<group>.json.    Built under /kaggle/working because /kaggle/input is read-only and DEIMv2 writes    alongside its data. Uses the HBB geometry -- DEIMv2 predicts upright boxes only, so the    OBB arm stays YOLO26-internal.    """    root = work / "deim_data"    (root / "annotations").mkdir(parents=True, exist_ok=True)    manifest = {}    import csv    with (data / "manifest.csv").open() as fh:        for r in csv.DictReader(fh):            manifest[r["id"]] = r    folds = json.loads((data / "folds.json").read_text())    id_of = {r["original"].replace(".jpg", ""): r["id"] for r in manifest.values()}    for g in GROUPS:        d = root / "images" / g        d.mkdir(parents=True, exist_ok=True)        for stem, grp in folds["photo_group"].items():            if grp != g:                continue            pid = id_of[stem]            src = data / "hbb" / "images" / f"{pid}.jpg"            dst = d / f"{pid}.jpg"            if not dst.exists():                shutil.copy2(src, dst)        shutil.copy2(data / "coco" / f"instances_{g}.json",                     root / "annotations" / f"instances_{g}.json")        n = len(list(d.glob("*.jpg")))        print(f"    {g}: {n} images")    # 3-fold CV means training on the OTHER TWO folds. COCO takes a single annotation file,    # so they are merged here -- with ids re-issued, because each per-fold json numbers its    # images from 1 and a naive concatenation would collide silently.    for k, held in enumerate(GROUPS[:3]):        others = [g for g in GROUPS[:3] if g != held]        d = root / "images" / f"train_fold{k}"        d.mkdir(parents=True, exist_ok=True)        merged = {"images": [], "annotations": [], "categories": None}        next_img, next_ann = 1, 1        for g in others:            js = json.loads((root / "annotations" / f"instances_{g}.json").read_text())            merged["categories"] = js["categories"]            remap = {}            for im in js["images"]:                remap[im["id"]] = next_img                merged["images"].append({**im, "id": next_img})                src = root / "images" / g / im["file_name"]                dst = d / im["file_name"]                if not dst.exists():                    shutil.copy2(src, dst)                next_img += 1            for an in js["annotations"]:                merged["annotations"].append({**an, "id": next_ann,                                              "image_id": remap[an["image_id"]]})                next_ann += 1        (root / "annotations" / f"instances_train_fold{k}.json").write_text(            json.dumps(merged, indent=1))        print(f"    train_fold{k}: {len(merged['images'])} images, "              f"{len(merged['annotations'])} annotations  (= {' + '.join(others)})")    return rootdef patch_repo(repo: Path):    """Apply the torchvision v2 transform rename if this torchvision needs it."""    import torchvision    tv = tuple(int(x) for x in torchvision.__version__.split(".")[:2])    print(f"  torchvision {torchvision.__version__}")    if tv < (0, 21):        print("    < 0.21, no transform patch needed")        return    n = 0    for p in repo.rglob("*.py"):        s = p.read_text()        o = s        s = re.sub(r"\bdef _get_params\b", "def make_params", s)        s = re.sub(r"\bdef _transform\b", "def transform", s)        s = re.sub(r"\bself\._get_params\b", "self.make_params", s)        s = re.sub(r"\bself\._transform\b", "self.transform", s)        if s != o:            p.write_text(s)            n += 1    print(f"    patched {n} file(s) for the torchvision >=0.21 v2 transform API (upstream PR #139)")def write_config(repo: Path, out: Path, variant: str, fold: str, data_root: Path,                 ckpt: Path, epochs: int, batch: int):    k = GROUPS.index(fold)    # Warm-up must be a small fraction of the run, not a fixed 2000. With 30 images at batch 4    # that is 8 iterations per epoch, so the COCO default would spend 250 epochs warming up and    # the learning rate would never reach its useful range (DEIM issue #5).    n_train = len(json.loads(        (data_root / "annotations" / f"instances_train_fold{k}.json").read_text())["images"])    iters_per_epoch = max(1, -(-n_train // batch))    warmup = max(20, int(0.05 * iters_per_epoch * epochs))    cfg = out / f"deimv2_{variant}_{fold}.yml"    cfg.write_text(f"""__include__: [ '{repo}/configs/deimv2/deimv2_hgnetv2_{variant}_coco.yml' ]# 3 gesture classes, 1-indexed in the COCO json, so the head needs 4 slotsnum_classes: 4remap_mscoco_category: Falsetrain_dataloader:  total_batch_size: {batch}          # 32 would give floor(30/32)=0 batches -> empty loader  dataset:    img_folder: {data_root}/images/train_fold{k}    ann_file: {data_root}/annotations/instances_train_fold{k}.json  drop_last: False                   # with 30 images, True discards the only partial batchval_dataloader:  total_batch_size: {batch}  dataset:    img_folder: {data_root}/images/{fold}    ann_file: {data_root}/annotations/instances_{fold}.jsonepoches: {epochs}lr_warmup:  warmup_iter: {warmup}                    # 5% of {iters_per_epoch * epochs} total iters                                     # ({iters_per_epoch} iters/epoch x {epochs} epochs)output_dir: {out}/runs/{variant}_{fold}tuning: {ckpt.resolve()}""")    return cfgdef main():    ap = argparse.ArgumentParser()    ap.add_argument("--data", type=Path, required=True)    ap.add_argument("--ckpt", type=Path, required=True)    ap.add_argument("--work", type=Path, default=Path("/kaggle/working"))    ap.add_argument("--variants", nargs="+", default=list(VARIANTS))    ap.add_argument("--epochs", type=int, default=120)    ap.add_argument("--batch", type=int, default=4)    ap.add_argument("--skip-clone", action="store_true")    args = ap.parse_args()    args.data = resolve_data(args.data)    repo = args.work / "DEIMv2"    print("=== 1. repository ===")    if not repo.exists() and not args.skip_clone:        run(["git", "clone", "--depth", "1", REPO_URL, str(repo)])    print(f"  {repo}  {'present' if repo.exists() else 'MISSING'}")    print("\n=== 2. dependencies ===")    print("  NOT running pip install -r requirements.txt (it pins torch==2.5.1)")    run([sys.executable, "-m", "pip", "install", "-q",         "faster-coco-eval", "calflops", "omegaconf", "loguru", "easydict"])    print("\n=== 3. torch / torchvision compatibility ===")    if repo.exists():        patch_repo(repo)    print("\n=== 4. data ===")    data_root = stage_data(args.data, args.work)    print("\n=== 5. configs ===")    cfg_dir = args.work / "deim_configs"    cfg_dir.mkdir(parents=True, exist_ok=True)    made = 0    for v in args.variants:        ck = (args.ckpt / VARIANTS[v]).resolve()        if not ck.exists():            print(f"  {v}: checkpoint missing ({ck.name}) - skipped")            continue        for fold in GROUPS[:3]:            write_config(repo, cfg_dir, v, fold, data_root, ck, args.epochs, args.batch)            made += 1    print(f"  wrote {made} config(s) -> {cfg_dir}")    print("\n=== next ===")    print(f"  cd {repo} && python train.py -c {cfg_dir}/deimv2_n_fold0.yml --use-amp --seed 42")    print("\n  Before trusting a run, assert the loader is non-empty:")    print("    len(train_dataloader) > 0     # DEIM issue #82")    print("    warm-up ends inside the first 10% of epochs")if __name__ == "__main__":    main()

In [ ]:
%%writefile /kaggle/working/run_deim.py#!/usr/bin/env python3"""Run the DEIMv2 arm: 8 variants x 3 folds, resumable, with a pre-flight guard.THE GUARD IS THE POINT. DEIMv2's COCO defaults produce an EMPTY dataloader at 30 images(total_batch_size 32, drop_last True -> floor(30/32) = 0 batches), and training then doesnothing while still printing epochs and exiting 0. Every config is checked for a non-emptyloader before a single run starts, so a silent no-op cannot be mistaken for a bad result.Results append to results_deim.jsonl after each run; completed runs are skipped on restart.Usage:    !python run_deim.py --repo /kaggle/working/DEIMv2 \                        --configs /kaggle/working/deim_configs --out /kaggle/working"""from __future__ import annotationsimport argparseimport jsonimport reimport subprocessimport sysimport timefrom pathlib import Pathdef preflight(cfg: Path) -> tuple[bool, str]:    """Read the config and reject settings that silently prevent training."""    text = cfg.read_text()    def get(key, default=None):        m = re.search(rf"^\s*{key}\s*:\s*(\S+)", text, re.M)        return m.group(1) if m else default    batch = int(get("total_batch_size", 32))    drop = (get("drop_last", "True") or "True").lower().startswith("t")    warm = int(get("warmup_iter", 2000))    epochs = int(get("epoches", 100))    ncls = int(get("num_classes", 80))    ann = re.search(r"ann_file:\s*(\S+)", text)    n_train = 0    if ann and Path(ann.group(1)).exists():        n_train = len(json.loads(Path(ann.group(1)).read_text())["images"])    problems = []    batches = n_train // batch if drop else -(-n_train // batch)    if batches == 0:        problems.append(f"EMPTY DATALOADER: {n_train} images, batch {batch}, "                        f"drop_last={drop} -> 0 batches")    if batches and warm > batches * epochs * 0.10:        problems.append(f"warmup_iter {warm} exceeds 10% of {batches*epochs} total iters")    if ncls < 4:        problems.append(f"num_classes {ncls}: categories are 1-indexed, so 3 classes need 4")    if n_train == 0:        problems.append("training annotation file missing or empty")    return (not problems), "; ".join(problems) or f"{n_train} imgs, {batches} batches/epoch"def prune_forever(run_dir: Path, keep: int, stop):    """Delete all but the newest `keep` checkpoints, every 20 s, while training runs.    DEIMv2 writes a checkpoint per epoch. At 120 epochs an `x` run alone would write ~70 GB,    and the post-run cleanup never gets a chance because the disk fills first. Two parallel    processes double the rate. Pruning on a timer is ugly but it is the only thing that works    without patching the repo's solver.    """    import time as _t    while not stop.is_set():        try:            ws = sorted(run_dir.rglob("*.pth"), key=lambda w: w.stat().st_mtime)            for w in ws[:-keep] if keep else ws:                try:                    w.unlink()                except OSError:                    pass        except Exception:            pass        stop.wait(20)def env_fingerprint():    """Record the machine and stack per run.    If any part of a sweep ends up on different hardware -- Kaggle T4 today, a rented 4090    tomorrow -- this is what lets you check whether a difference is the model or the machine.    Training hardware does not change a model to within fold-to-fold noise, but that is a    claim the data should support rather than one the methods section merely asserts.    """    import torch, platform    return {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",            "torch": torch.__version__, "python": platform.python_version(),            "host": platform.node()}def parse_ap(log: str):    """Best COCO AP over the run. DEIMv2 prints the standard 12-line COCOeval block."""    best = {}    for m in re.finditer(r"Average Precision.*?IoU=0\.50:0\.95.*?area=\s*all.*?\]\s*=\s*([\d.\-]+)", log):        try:            v = float(m.group(1))            if v >= 0:                best["ap50_95"] = max(best.get("ap50_95", 0.0), v)        except ValueError:            pass    for m in re.finditer(r"Average Precision.*?IoU=0\.50\s.*?area=\s*all.*?\]\s*=\s*([\d.\-]+)", log):        try:            v = float(m.group(1))            if v >= 0:                best["ap50"] = max(best.get("ap50", 0.0), v)        except ValueError:            pass    return bestdef main():    ap = argparse.ArgumentParser()    ap.add_argument("--repo", type=Path, required=True)    ap.add_argument("--configs", type=Path, required=True)    ap.add_argument("--out", type=Path, default=Path("/kaggle/working"))    ap.add_argument("--variants", nargs="+", default=None)    ap.add_argument("--check-only", action="store_true")    ap.add_argument("--results", type=Path, default=None,                    help="results file; give each parallel process its own, then merge")    ap.add_argument("--port-base", type=int, default=29500,                    help="torchrun rendezvous port; two processes must not share one")    ap.add_argument("--keep-checkpoints", action="store_true",                    help="keep the best checkpoint per run (needed for confusion matrices)")    args = ap.parse_args()    cfgs = sorted(args.configs.glob("deimv2_*.yml"))    if args.variants:        cfgs = [c for c in cfgs if c.stem.split("_")[1] in args.variants]    if not cfgs:        raise SystemExit(f"no configs in {args.configs}")    print(f"=== pre-flight: {len(cfgs)} config(s) ===")    ok_cfgs = []    for c in cfgs:        ok, msg = preflight(c)        print(f"  {'OK  ' if ok else 'FAIL'} {c.stem:28s} {msg}")        if ok:            ok_cfgs.append(c)    if not ok_cfgs:        raise SystemExit("\nno config passed pre-flight; fix setup_deimv2.py before training")    if args.check_only:        return    results = args.results or (args.out / "results_deim.jsonl")    done = set()    if results.exists():        for line in results.read_text().splitlines():            try:                r = json.loads(line)                if "error" not in r:          # a failed run must not block a retry                    done.add(r["config"])            except Exception:                pass    todo = [c for c in ok_cfgs if c.stem not in done]    print(f"\n{len(done)} done, {len(todo)} to run\n")    for i, cfg in enumerate(todo, 1):        _, variant, fold = cfg.stem.split("_", 2)        print(f"[{i}/{len(todo)}] {cfg.stem}")        t0 = time.time()        # DEIMv2 calls init_process_group(backend="nccl") unconditionally -- there is no        # single-GPU branch -- so `python train.py` raises before the first step. torchrun        # sets the rendezvous env vars even for one process. A distinct port per run avoids        # "address already in use" when the previous socket has not been released yet.        port = args.port_base + (i % 100)        run_dir = args.configs / "runs" / f"{variant}_{fold}"        run_dir.mkdir(parents=True, exist_ok=True)        import threading        stop = threading.Event()        pruner = threading.Thread(target=prune_forever,                                  args=(run_dir, 2 if args.keep_checkpoints else 1, stop),                                  daemon=True)        pruner.start()        proc = subprocess.run(            ["torchrun", "--nproc_per_node=1", f"--master_port={port}",             "train.py", "-c", str(cfg), "--use-amp", "--seed", "42"],            cwd=args.repo, capture_output=True, text=True)        stop.set(); pruner.join(timeout=5)        if not args.keep_checkpoints:            for w in run_dir.rglob("*.pth"):                w.unlink()        log = proc.stdout + proc.stderr        (args.out / "logs").mkdir(exist_ok=True)        (args.out / "logs" / f"{cfg.stem}.log").write_text(log)        rec = {"config": cfg.stem, "variant": variant, "fold": fold,               "minutes": round((time.time() - t0) / 60, 2), "returncode": proc.returncode,               "env": env_fingerprint()}        rec.update(parse_ap(log))        if proc.returncode != 0:            rec["error"] = log.strip().splitlines()[-1][:300] if log.strip() else "no output"            print(f"   FAILED rc={proc.returncode}: {rec['error'][:120]}")        else:            print(f"   AP50-95 {rec.get('ap50_95','?')}  AP50 {rec.get('ap50','?')}  "                  f"{rec['minutes']:.1f} min")        with results.open("a") as fh:            fh.write(json.dumps(rec) + "\n")    print(f"\nresults -> {results}\nlogs    -> {args.out/'logs'}")if __name__ == "__main__":    main()

## 4 · YOLO26 — 30 runsBoth geometries, five sizes, three folds. Appends to `results.jsonl` after every run, so aninterrupted session resumes rather than restarting. Re-run this cell as often as you like.

In [ ]:
!cd /kaggle/working && python train_yolo.py \    --data {DATA} --out /kaggle/working --epochs 100 --imgsz 640 --batch 8

### 4b · Resolution sweep (optional)Accuracy at 320 and 416 for the smallest model, so the latency arm has matching accuracynumbers. Input size is the largest single lever on inference time.

In [ ]:
!cd /kaggle/working && python train_yolo.py \    --data {DATA} --out /kaggle/working --sizes n --epochs 100 --imgsz-sweep

## 5 · DEIMv2 — setupClones the repo, patches the torchvision v2 transform API if needed, stages a COCO tree under`/kaggle/working` (the input mount is read-only), and writes 24 configs.**`requirements.txt` is deliberately not installed** — it pins `torch==2.5.1`, which wouldreplace Kaggle's CUDA-matched build and break every GPU operation.

In [ ]:
!cd /kaggle/working && python setup_deimv2.py \    --data {DATA} --ckpt {CKPT} --work /kaggle/working --epochs 120 --batch 4

### 5b · Pre-flightDEIMv2's COCO defaults give an **empty dataloader** at 30 images — `total_batch_size 32` with`drop_last: True` is `floor(30/32) = 0` batches — and training then does nothing while stillprinting epochs and exiting cleanly. This checks every config before any run starts.

In [ ]:
!cd /kaggle/working && python run_deim.py \    --repo /kaggle/working/DEIMv2 --configs /kaggle/working/deim_configs --check-only

### 5c · DEIMv2 — 24 runsStart with `pico` and `n`; the DINOv3 variants are larger and slower.

In [ ]:
!cd /kaggle/working && python run_deim.py \    --repo /kaggle/working/DEIMv2 --configs /kaggle/working/deim_configs \    --out /kaggle/working --variants pico n

In [ ]:
# the rest, once the first two are known good!cd /kaggle/working && python run_deim.py \    --repo /kaggle/working/DEIMv2 --configs /kaggle/working/deim_configs --out /kaggle/working

## 6 · ResultsCross-validation reported as mean ± SD across the three folds (ddof=1).

In [ ]:
import json, statistics, collectionsfrom pathlib import Pathrows = []p = Path("/kaggle/working/results.jsonl")if p.exists():    for line in p.read_text().splitlines():        if line.strip():            rows.append(json.loads(line))ok = [r for r in rows if "error" not in r]print(f"{len(ok)} successful runs, {len(rows)-len(ok)} failed\n")agg = collections.defaultdict(list)for r in ok:    agg[(r["geom"], r["size"], r["imgsz"])].append(r["mAP50-95"])print(f"{'geom':5s} {'size':5s} {'imgsz':>6s} {'n':>2s} {'mAP50-95':>10s} {'SD':>8s}")print("-" * 46)for k in sorted(agg):    v = agg[k]    sd = statistics.stdev(v) if len(v) > 1 else 0.0   # ddof=1: with n=3 the population SD    print(f"{k[0]:5s} {k[1]:5s} {k[2]:6d} {len(v):2d} "   # understates the spread by ~18%          f"{statistics.mean(v):10.4f} {sd:8.4f}")d = Path("/kaggle/working/results_deim.jsonl")if d.exists():    print("\nDEIMv2")    dag = collections.defaultdict(list)    for line in d.read_text().splitlines():        if line.strip():            r = json.loads(line)            if "ap50_95" in r:                dag[r["variant"]].append(r["ap50_95"])    for k in sorted(dag):        v = dag[k]        sd = statistics.stdev(v) if len(v) > 1 else 0.0        print(f"  {k:6s} n={len(v)}  AP50-95 {statistics.mean(v):.4f} +/- {sd:.4f}")

## 7 · Collect the outputsCheckpoints come home so latency can be measured on the target laptop — a datacentre GPUanswers a question nobody asked.

In [ ]:
import shutil, osfrom pathlib import Pathout = Path("/kaggle/working/collected"); out.mkdir(exist_ok=True)n = 0for w in Path("/kaggle/working/runs").rglob("weights/best.pt"):    dst = out / f"{w.parent.parent.name}.pt"    shutil.copy2(w, dst); n += 1for f in ("results.jsonl", "results_deim.jsonl"):    p = Path("/kaggle/working") / f    if p.exists():        shutil.copy2(p, out / f)print(f"{n} checkpoints + results -> {out}")print(f"total {sum(f.stat().st_size for f in out.rglob('*') if f.is_file())/1e6:.1f} MB")

## Notes**Use Save Version → Save & Run All (Commit)** for the full sweep. Interactive sessions die at12 hours and also when the browser disconnects; a committed run survives both.**Download afterwards:**```kaggle kernels output shumahara/<notebook-slug> -p ./results```**If you re-upload the dataset**, bump the version in the Input panel. The notebook otherwisekeeps using the version it was attached to, with no warning — a notebook quietly training onlast week's labels is the most common Kaggle mistake.**Not done here:** the 4x4 confusion matrix, per-hand AUC-ROC, and Core ML export. Those runlocally on the M4 against the checkpoints collected above, because the latency question isabout that machine.